# 02 · El objetivo: razón adjudicado / referencial

**Objetivo:** entender la variable que predice el modelo antes de entrenarlo.

El modelo estima `log(monto adjudicado / valor referencial)`; el monto esperado es
`valor referencial x exp(predicción)`. Predecir la razón en lugar del monto no pierde nada (el
error en escala log es el mismo) y concentra el modelo en lo que es incierto: el descuento o el
sobreprecio frente al referencial.

**Entradas:** la capa gold (`data/curated`) y `config/model_config.yaml`.

In [ ]:
# Configuración del entorno (ejecutar primero)
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd

# Raíz del proyecto: la carpeta actual o la primera superior con pyproject.toml.
CWD = Path.cwd().resolve()
ROOT = next((p for p in (CWD, *CWD.parents) if (p / "pyproject.toml").is_file()), CWD)
sys.path.insert(0, str(ROOT / "src"))  # permite importar el paquete aunque no esté instalado

CURATED = ROOT / "data" / "curated"          # capa gold
FEATURES = ROOT / "data" / "features"        # dataset de modelado
PREDICTIONS = ROOT / "data" / "predictions"  # monto esperado y atípicos
MODELS = ROOT / "models"                     # modelos entrenados
RAW_API = ROOT / "data" / "raw" / "api"      # páginas crudas de la API del OECE
if not (CURATED / "fact_proceso.parquet").exists():
    raise FileNotFoundError(
        "No hay datos procesados. Ejecuta primero en la terminal: contrataciones run --source sample"
    )

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:,.4f}".format)
PALETA = ["#1F2A44", "#3E6A8A", "#7FA7C2", "#C9A227", "#B5533C", "#5B8C5A", "#8C8C8C"]
COLORES_DESVIACION = {
    "Normal": "#C8CFD8",
    "Sobrecosto inusual": "#B5533C",
    "Subcosto inusual": "#3E6A8A",
    "Revisar datos": "#C9A227",
}
plt.rcParams.update({
    "figure.figsize": (10, 4.5), "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.3, "axes.prop_cycle": plt.cycler(color=PALETA),
    "axes.titleweight": "bold", "axes.titlesize": 12,
})
porcentaje = mticker.PercentFormatter(1.0)
print("Proyecto:", ROOT)

In [ ]:
from contrataciones.features.builder import build_features
from contrataciones.features.catalog import load_feature_specs
from contrataciones.features.dataset import TARGET, build_modeling_table, load_gold, population_summary
from contrataciones.models.trainer import assign_partitions, split_dates
from contrataciones.utils.config_loader import load_config

config = load_config("model_config")
gold = load_gold(CURATED)
specs = load_feature_specs(config["features"])
table = build_modeling_table(gold, config["population"])
X = build_features(table, specs, config["entity_types"])
poblacion = table[table["es_poblacion"]]
population_summary(table)

## 1. Población del modelo

Entran los procesos con fecha de convocatoria, adjudicación válida y valor referencial. Los de
razón fuera del rango plausible se evalúan (como *Revisar datos*) pero no entrenan.

In [ ]:
motivos = table["motivo_exclusion"].fillna("Entrenable").value_counts()
ax = motivos.sort_values().plot.barh(title="Procesos por motivo de exclusión", color=PALETA[1])
ax.set_ylabel(""); plt.show()
motivos

## 2. Distribución de la razón adjudicado / referencial

In [ ]:
razon = poblacion["ratio_adjudicado_referencial"]
ax = razon.clip(0, 1.5).plot.hist(bins=60, title="Razón adjudicado / referencial (recortada en 1.5)")
ax.axvline(1.0, color=PALETA[4], linestyle="--", linewidth=1)
ax.set_xlabel("adjudicado / referencial"); plt.show()
print(f"Adjudicado = referencial al céntimo: {poblacion['es_adjudicado_igual_referencial'].mean():.1%}")
razon.describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95])

## 3. Por método y categoría

La mediana por método y categoría es exactamente la **línea base B1**: el modelo tiene que
mejorarla para que valga la pena.

In [ ]:
datos = pd.concat([poblacion[TARGET], X.loc[poblacion.index, ["metodo", "categoria"]]], axis=1)
principales = datos["metodo"].value_counts().head(8).index
datos[datos["metodo"].isin(principales)].boxplot(column=TARGET, by="metodo", rot=20, figsize=(10, 5))
plt.suptitle(""); plt.title("log(razón) por procedimiento"); plt.xlabel(""); plt.ylabel("log(adjudicado / referencial)")
plt.show()
datos.groupby(["metodo", "categoria"])[TARGET].agg(["size", "median"]).sort_values("size", ascending=False).head(15)

## 4. Tamaño del proceso

¿Los procesos grandes logran más descuento? El modelo puede captarlo con `log_valor_referencial`.

In [ ]:
plt.scatter(X.loc[poblacion.index, "log_valor_referencial"], poblacion[TARGET], s=6, alpha=0.4)
plt.title("log(razón) frente al tamaño del proceso")
plt.xlabel("log(1 + valor referencial)"); plt.ylabel("log(razón)"); plt.show()

## 5. Cobertura temporal y división

La evaluación es temporal: se entrena con procesos convocados (y adjudicados) antes de la
validación, y las convocatorias más recientes no se evalúan porque solo los procesos rápidos ya
tienen buena pro.

In [ ]:
fechas = split_dates(table, config["split"])
particiones = assign_partitions(table, fechas)
mensual = table[table["es_entrenable"]].groupby(table["fecha_convocatoria"].dt.to_period("M")).size()
ax = mensual.plot.bar(title="Procesos entrenables por mes de convocatoria", color=PALETA[2], width=0.9)
etiquetas = [str(p) for p in mensual.index]
for nombre, fecha in [("validación", fechas.inicio_validacion), ("prueba", fechas.inicio_prueba),
                      ("corte de madurez", fechas.corte_madurez)]:
    periodo = str(pd.Period(fecha, "M"))
    if periodo in etiquetas:
        ax.axvline(etiquetas.index(periodo) - 0.5, color=PALETA[4], linestyle="--")
        ax.annotate(nombre, (etiquetas.index(periodo) - 0.4, ax.get_ylim()[1] * 0.92), fontsize=9)
ax.set_xticks(range(0, len(etiquetas), 3), etiquetas[::3], rotation=0); ax.set_xlabel(""); plt.show()
print(fechas.to_dict())
particiones.value_counts()

## Conclusiones

* Con los **datos sintéticos** la razón es aleatoria e independiente de todas las variables: no
  hay nada que aprender y el modelo no debería superar a la línea base (es el resultado esperado).
* Con **datos reales**, busca estructura: diferencias por procedimiento y categoría, efecto del
  tamaño y una masa de procesos con razón exactamente 1 (ver `01_auditoria_fuga_de_datos.ipynb`).
* Siguiente paso: `contrataciones-ml train` y `03_modelos_y_baselines.ipynb`.